In [1]:
from datasets import load_from_disk
from pathlib import Path
import os


/mnt/data/envs/finetune-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(os.listdir("../data"))
print(os.listdir("../data/processed"))


['format_for_sft.ipynb', 'processed', 'raw']
['train', 'control_spa', 'sft', 'val', 'test', 'control_por']


In [3]:
train_ds = load_from_disk("../data/processed/train")
val_ds = load_from_disk("../data/processed/val")
test_ds = load_from_disk("../data/processed/test")
por_ds = load_from_disk("../data/processed/control_spa")
spa_ds = load_from_disk("../data/processed/control_por")


In [4]:
sample = test_ds[0]
for key, value in sample.items():
    print(key, ":", value)
print(test_ds.column_names)


inputs : What is the significance of time-series analysis in data science, and where is it commonly applied?
targets : Time-series analysis is crucial for understanding temporal patterns in data. It is commonly applied in finance, forecasting, and IoT applications to analyze trends over time.
language : English
language_code : eng
annotation_type : original-annotations
user_id : 7e2f92b1fdb1a83cbd6d507fff9c5478fee7da855370d4644984399159bbf852
['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id']


### Formatting function

In [5]:
def format_example(example):
    return {
        "messages": [
            {"role": "user", "content": example["inputs"]},
            {"role": "assistant", "content": example["targets"]},
        ],
        "language_code": example["language_code"],
        "language": example["language"],
    }


In [6]:
original_columns = train_ds.column_names

formatted_train = train_ds.map(format_example, remove_columns=original_columns)
formatted_val = val_ds.map(format_example, remove_columns=original_columns)
formatted_test = test_ds.map(format_example, remove_columns=original_columns)
formatted_por_ds = por_ds.map(format_example, remove_columns=original_columns)
formatted_spa_ds = spa_ds.map(format_example, remove_columns=original_columns)


Map: 100%|██████████| 300/300 [00:00<00:00, 15367.88 examples/s]


In [7]:
for code in ["eng", "fra", "ita"]:
    subset = formatted_train.filter(lambda x: x["language_code"] == code)
    print(f"\n===== {code} =====")
    sample = subset[0]
    print(sample["messages"][0]["content"])
    print("-" * 40)
    print(sample["messages"][1]["content"])
    print("=" * 80)


Filter: 100%|██████████| 4882/4882 [00:00<00:00, 36281.38 examples/s]



===== eng =====
What is a data structure?
----------------------------------------
A data structure is a way of organizing and storing data to perform operations efficiently. It defines the relationships and functions that can be applied to the data.


Filter: 100%|██████████| 4882/4882 [00:00<00:00, 80953.07 examples/s]



===== fra =====
De quoi sont faits les anneaux de Saturne ?
----------------------------------------
Ils sont constitués de blocs de glace et de roches dont la taille varie entre celle d'un grain de sable et celle d'une montagne. Les anneaux de Saturne s'étendent sur plus de 500 000 kilomètres de large, mais sur moins de 1 kilomètre d'épaisseur. Beaucoup plus jeunes que Saturne elle-même, ils se seraient formés il y a quelques centaines de millions d'années, quand un ou plusieurs satellites auraient explosé, broyés par la forte gravité de la planète.


Filter: 100%|██████████| 4882/4882 [00:00<00:00, 83244.95 examples/s]


===== ita =====
L'Australia è il terzo Paese del mondo per estensione. Vero o falso?
----------------------------------------
La frase sopra è falsa.


In [8]:
print(formatted_train.column_names)
print(formatted_val.column_names)


['language', 'language_code', 'messages']
['language', 'language_code', 'messages']


In [9]:
formatted_train.save_to_disk("../data/processed/sft/train_sft")
formatted_val.save_to_disk("../data/processed/sft/validation_sft")
formatted_test.save_to_disk("../data/processed/sft/test_sft")
formatted_por_ds.save_to_disk("../data/processed/sft/probe_spa_sft")
formatted_spa_ds.save_to_disk("../data/processed/sft/probe_por_sft")


Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 71668.92 examples/s]
